In [1]:
## Tier 3 — IsolationForest on entity features
#After Tier 1 MVP + Tier 2 (z>2 or five_pct≥90).
#Input for ML: `t2_clean` — residual panel after rules + store-month filter.

#sum:
#Убрано всего: 59 368 (6.17%)
#entity_key известен у 99.56% — для Tier 3 это очень хорошо (лучше, чем ожидали; после Tier1+2 почти все строки идентифицируемы)

In [2]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()
from db.connection import get_engine
from fraud_guard.tier1 import (
    Tier1Config,
    apply_tier1,
    filter_answered_metric_rows,
    keep_clean_rows,
    top_box_rate,
    build_entity_key,
)
from fraud_guard.tier2 import apply_tier2, keep_clean_rows as keep_clean_rows_tier2

engine = get_engine()
VIEW = "dbo.TargetsByMetrics_RateGetAnswers"

df = pd.read_sql(f"SELECT * FROM {VIEW}", engine)
print("raw shape:", df.shape)

work = filter_answered_metric_rows(df)
t1_flagged = apply_tier1(work, config=Tier1Config(enable_always_topbox=False))
t1_clean = keep_clean_rows(t1_flagged)

t2_flagged = apply_tier2(t1_clean)
t2_clean = keep_clean_rows_tier2(t2_flagged)

# entity coverage on residual (important for Tier 3)
t2_clean = t2_clean.copy()
t2_clean["entity_key"] = build_entity_key(t2_clean)
known_entity = t2_clean["entity_key"].notna().sum()

print("\n--- pipeline baseline ---")
print(f"0 answered Q10012: {len(work):>8}  top-box: {top_box_rate(work):.4f}")
print(f"1 after Tier1 MVP: {len(t1_clean):>8}  top-box: {top_box_rate(t1_clean):.4f}")
print(f"2 after Tier2:     {len(t2_clean):>8}  top-box: {top_box_rate(t2_clean):.4f}")

print(f"\ndropped Tier1: {len(work) - len(t1_clean):>8}  ({100*(len(work)-len(t1_clean))/len(work):.2f}%)")
print(f"dropped Tier2: {len(t1_clean) - len(t2_clean):>8}  ({100*(len(t1_clean)-len(t2_clean))/len(t1_clean):.2f}%)")
print(f"total dropped:   {len(work) - len(t2_clean):>8}  ({100*(len(work)-len(t2_clean))/len(work):.2f}%)")

print(f"\nentity_key known in t2_clean: {known_entity:,} / {len(t2_clean):,}  ({100*known_entity/len(t2_clean):.2f}%)")

raw shape: (1218050, 35)

--- pipeline baseline ---
0 answered Q10012:   962799  top-box: 0.6813
1 after Tier1 MVP:   928487  top-box: 0.6715
2 after Tier2:       903430  top-box: 0.6644

dropped Tier1:    34312  (3.56%)
dropped Tier2:    25057  (2.70%)
total dropped:      59369  (6.17%)

entity_key known in t2_clean: 899,497 / 903,430  (99.56%)


In [3]:
## Шаг B — feature table на уровне entity_key без .apply()
#Идея: для каждого клиента считаем профиль поведения (адаптация твоего [orders_per_id, time_to_review, conversion_rate, score_variance]).

In [5]:
import numpy as np

eligible = t2_clean[t2_clean["entity_key"].notna()].copy()

eligible["latency_min"] = (
    pd.to_datetime(eligible["AnswerTime"], errors="coerce")
    - pd.to_datetime(eligible["PrintDateTime"], errors="coerce")
).dt.total_seconds() / 60.0
eligible["answer_day"] = pd.to_datetime(eligible["AnswerTime"], errors="coerce").dt.date
eligible["is_topbox"] = eligible["Answer_Value"].eq(5).astype(int)

g = eligible.groupby("entity_key", sort=False)

feat = g.agg(
    n_answers=("Answer_Value", "count"),
    topbox_rate=("is_topbox", "mean"),
    score_variance=("Answer_Value", "var"),
    latency_median_min=("latency_min", "median"),
    latency_std_min=("latency_min", "std"),
    n_stores=("PrintStore", "nunique"),
).reset_index()

# max answers same entity × store × day (Tier1-like signal at entity level)
store_day_cnt = (
    eligible.groupby(["entity_key", "PrintStore", "answer_day"], sort=False)
    .size()
    .groupby(level=0)
    .max()
    .rename("max_answers_store_day")
)
feat = feat.join(store_day_cnt, on="entity_key")

# n distinct store-days
store_day_n = (
    eligible.groupby(["entity_key", "PrintStore", "answer_day"], sort=False)
    .size()
    .reset_index(name="_")
    .groupby("entity_key", sort=False)
    .size()
    .rename("n_store_days")
)
feat = feat.join(store_day_n, on="entity_key")

feat["score_variance"] = feat["score_variance"].fillna(0.0)
feat["latency_std_min"] = feat["latency_std_min"].fillna(0.0)

print("entities total:", len(feat))
print(feat.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3))

MIN_N = 3
feat_if = feat[feat["n_answers"] >= MIN_N].copy()
print(f"\nentities with n>={MIN_N}:", len(feat_if), f"({100*len(feat_if)/len(feat):.2f}%)")

rows_covered = eligible["entity_key"].isin(set(feat_if["entity_key"])).sum()
print(f"answer rows covered (n>={MIN_N}):", rows_covered, f"({100*rows_covered/len(eligible):.2f}%)")

always5 = feat_if[feat_if["topbox_rate"] >= 1.0]
print(f"\nalways-5 entities (n>={MIN_N}):", len(always5))
print("top max_answers_store_day:")
print(
    feat_if.nlargest(10, "max_answers_store_day")[
        ["entity_key", "n_answers", "topbox_rate", "n_stores", "max_answers_store_day"]
    ].to_string(index=False)
)

entities total: 276330
        n_answers  topbox_rate  score_variance  latency_median_min  \
count  276330.000   276330.000      276330.000          276228.000   
mean        3.255        0.691           0.295              28.419   
std        10.510        0.422           0.897              15.074   
min         1.000        0.000           0.000               5.050   
50%         1.000        1.000           0.000              25.667   
90%         7.000        1.000           0.917              48.650   
95%        13.000        1.000           2.000              57.775   
99%        30.000        1.000           4.500              73.267   
max      4322.000        1.000           8.000            1177.150   

       latency_std_min    n_stores  max_answers_store_day  n_store_days  
count       276330.000  276330.000             276325.000    276325.000  
mean             4.531       1.686                  1.026         3.208  
std              7.900       1.660                  0.

In [ ]:
#Суммаризация Step B
#Метрика	Значение	Смысл
#Уникальных entity
#276 330
#После Tier1+2
#Медиана ответов
#1
#Большинство — разовые клиенты
#Entity с n≥3
#69 266 (25%)
#Пул для IF
#Строк под IF
#659 999 (73%)
#Основная масса ответов
#Always-5 при n≥3
#19 850
#Главные кандидаты на fraud

In [ ]:
## Шаг C — IsolationForest + sweep contamination

In [6]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    "n_answers",
    "topbox_rate",
    "score_variance",
    "latency_median_min",
    "latency_std_min",
    "n_stores",
    "max_answers_store_day",
    "n_store_days",
]

X = feat_if[FEATURE_COLS].copy()
# log-scale heavy tail on counts
X["n_answers"] = np.log1p(X["n_answers"])
X["n_store_days"] = np.log1p(X["n_store_days"])
X["n_stores"] = np.log1p(X["n_stores"])
X = X.fillna(X.median())

X_scaled = StandardScaler().fit_transform(X)

bad_entities_by_c: dict[float, set[str]] = {}

print(f"{'contam':>8} {'entities_flagged':>16} {'rows_flagged':>12} {'pct_rows':>9} {'topbox_drop':>12} {'topbox_after':>13} {'delta_pp':>9}")
print("-" * 95)

baseline_tb = top_box_rate(t2_clean)

for contam in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]:
    iso = IsolationForest(
        n_estimators=200,
        contamination=contam,
        random_state=42,
        n_jobs=-1,
    )
    pred = iso.fit_predict(X_scaled)  # -1 = anomaly
    flagged_entities = set(feat_if.loc[pred == -1, "entity_key"])

    row_mask = eligible["entity_key"].isin(flagged_entities)
    dropped = eligible.loc[row_mask]
    kept = eligible.loc[~row_mask]

    # rows outside IF pool (n<3 entities) always kept
    n_if_pool = eligible["entity_key"].isin(set(feat_if["entity_key"]))
    kept_all = pd.concat([
        kept,
        eligible.loc[~n_if_pool],
    ])

    tb_drop = top_box_rate(dropped) if len(dropped) else float("nan")
    tb_after = top_box_rate(kept_all)
    delta_pp = 100 * (tb_after - baseline_tb)

    bad_entities_by_c[contam] = flagged_entities

    print(
        f"{contam:8.3f} {len(flagged_entities):16,d} {len(dropped):12,d} "
        f"{100*len(dropped)/len(eligible):8.2f}% {tb_drop:12.4f} {tb_after:13.4f} {delta_pp:+8.3f}"
    )

# pick working contamination for detail (adjust after you see table)
CONTAM = 0.02
bad_e = bad_entities_by_c[CONTAM]
print(f"\n--- detail @ contamination={CONTAM} ---")
print("flagged entities:", len(bad_e))
dropped_rows = eligible[eligible["entity_key"].isin(bad_e)]
print("dropped rows:", len(dropped_rows), f"({100*len(dropped_rows)/len(eligible):.2f}%)")
print("top-box dropped:", round(top_box_rate(dropped_rows), 4))
print("top-box after IF:", round(top_box_rate(
    pd.concat([eligible[~eligible["entity_key"].isin(bad_e)], eligible[~eligible["entity_key"].isin(set(feat_if["entity_key"]))]])
), 4))

# overlap with always-5 pool
always5_keys = set(feat_if.loc[feat_if["topbox_rate"] >= 1.0, "entity_key"])
print(f"overlap with always-5 (n>=3): {len(bad_e & always5_keys)} / {len(always5_keys)}")

# top flagged by volume
flagged_feat = feat_if[feat_if["entity_key"].isin(bad_e)].nlargest(15, "n_answers")
print("\ntop flagged entities:")
print(flagged_feat[["entity_key", "n_answers", "topbox_rate", "n_stores", "n_store_days"]].to_string(index=False))

  contam entities_flagged rows_flagged  pct_rows  topbox_drop  topbox_after  delta_pp
-----------------------------------------------------------------------------------------------
   0.005              347       25,564     2.84%       0.7640        0.6701   +0.567
   0.010              693       41,390     4.60%       0.7521        0.6692   +0.476
   0.020            1,386       61,670     6.86%       0.7047        0.6703   +0.591
   0.030            2,078       78,910     8.77%       0.6894        0.6709   +0.649
   0.050            3,464      106,433    11.83%       0.6796        0.6714   +0.701
   0.080            5,542      141,317    15.71%       0.6664        0.6730   +0.860
   0.100            6,927      160,450    17.84%       0.6602        0.6742   +0.974

--- detail @ contamination=0.02 ---
flagged entities: 1386
dropped rows: 61670 (6.86%)
top-box dropped: 0.7047
top-box after IF: 0.6703
overlap with always-5 (n>=3): 251 / 19850

top flagged entities:
  entity_key  n_answe

In [7]:
##sum:
#IsolationForest на entity-фичах (n≥3) находит high-volume аномалии (до 4322 ответов / 223 store). Удалённые строки ~70–76% top-box. Реальный эффект @2% contamination ≈ −0.25 п.п. к Tier2 (~61k строк). Overlap с always-5 минимален — IF дополняет Tier1+2, а не повторяет optional rule 3.

In [ ]:
## Шаг D — корректный pipeline + overlap

In [8]:
def tier3_clean_rows(df: pd.DataFrame, bad_entities: set[str]) -> pd.DataFrame:
    """Drop all rows belonging to flagged entities; unknown entity / n<3 kept."""
    out = df.copy()
    if "entity_key" not in out.columns:
        out["entity_key"] = build_entity_key(out)
    return out[~out["entity_key"].isin(bad_entities)].copy()

def pipeline_row(name: str, n: int, tb: float) -> None:
    print(f"{name:<22} {n:>9,}  top-box: {tb:.4f}  ({100*tb:.2f}%)")

# pick contamination after reviewing sweep (try 0.005 and 0.02)
for CONTAM in [0.005, 0.02]:
    bad_e = bad_entities_by_c[CONTAM]
    t3_clean = tier3_clean_rows(t2_clean, bad_e)

    print(f"\n=== Full pipeline @ contamination={CONTAM} ===")
    pipeline_row("0 answered Q10012", len(work), top_box_rate(work))
    pipeline_row("1 Tier1 MVP", len(t1_clean), top_box_rate(t1_clean))
    pipeline_row("2 Tier2", len(t2_clean), top_box_rate(t2_clean))
    pipeline_row("3 Tier3 (IF)", len(t3_clean), top_box_rate(t3_clean))

    dropped_t3 = len(t2_clean) - len(t3_clean)
    print(f"\nTier3 dropped: {dropped_t3:,} ({100*dropped_t3/len(t2_clean):.2f}% of Tier2 panel)")
    if dropped_t3:
        dropped_df = t2_clean[t2_clean["entity_key"].isin(bad_e) if "entity_key" in t2_clean.columns
                              else tier3_clean_rows(t2_clean, set())]  # fallback
        # safer:
        t2k = t2_clean.copy()
        t2k["entity_key"] = build_entity_key(t2k)
        dropped_df = t2k[t2k["entity_key"].isin(bad_e)]
        print(f"top-box dropped by Tier3: {top_box_rate(dropped_df):.4f}")

    print(f"delta Tier2→Tier3: {100*(top_box_rate(t3_clean)-top_box_rate(t2_clean)):+.3f} pp")
    print(f"delta raw→Tier3:   {100*(top_box_rate(t3_clean)-top_box_rate(work)):+.3f} pp")

# --- overlap: Tier3 vs Tier2 store-months ---
t2k = t2_clean.copy()
t2k["entity_key"] = build_entity_key(t2k)
high_sm = set(zip(
    t2_flagged.loc[t2_flagged["fraud_tier2_flag"], "PrintStore"],
    t2_flagged.loc[t2_flagged["fraud_tier2_flag"], "Year"],
    t2_flagged.loc[t2_flagged["fraud_tier2_flag"], "Month"],
))
t2k["_sm"] = list(zip(t2k["PrintStore"], t2k["Year"], t2k["Month"]))

CONTAM = 0.02
bad_e = bad_entities_by_c[CONTAM]
t3_rows = t2k[t2k["entity_key"].isin(bad_e)]
t2_only_rows = t2k[t2k["_sm"].isin(high_sm)]

print("\n=== Overlap Tier3 vs Tier2 ===")
print("rows in Tier2-high store-months:", len(t2_only_rows))
print("rows flagged by Tier3 @0.02:", len(t3_rows))
both = t3_rows.index.intersection(t2_only_rows.index)
print("rows caught by BOTH:", len(both), f"({100*len(both)/max(len(t3_rows),1):.1f}% of Tier3 drops)")
print("Tier3-only rows (not Tier2 SM):", len(t3_rows) - len(both))

# entity c:1 quick check (mask phone in output)
mega = feat_if[feat_if["n_answers"] == feat_if["n_answers"].max()].iloc[0]
print("\nmega-entity profile:")
print(mega.to_string())
print("rows if removed alone:", int((t2k["entity_key"] == mega["entity_key"]).sum()))


=== Full pipeline @ contamination=0.005 ===
0 answered Q10012        962,799  top-box: 0.6813  (68.13%)
1 Tier1 MVP              928,487  top-box: 0.6715  (67.15%)
2 Tier2                  903,430  top-box: 0.6644  (66.44%)
3 Tier3 (IF)             877,866  top-box: 0.6615  (66.15%)

Tier3 dropped: 25,564 (2.83% of Tier2 panel)
top-box dropped by Tier3: 0.7640
delta Tier2→Tier3: -0.290 pp
delta raw→Tier3:   -1.974 pp

=== Full pipeline @ contamination=0.02 ===
0 answered Q10012        962,799  top-box: 0.6813  (68.13%)
1 Tier1 MVP              928,487  top-box: 0.6715  (67.15%)
2 Tier2                  903,430  top-box: 0.6644  (66.44%)
3 Tier3 (IF)             841,760  top-box: 0.6615  (66.15%)

Tier3 dropped: 61,670 (6.83% of Tier2 panel)
top-box dropped by Tier3: 0.7047
delta Tier2→Tier3: -0.295 pp
delta raw→Tier3:   -1.979 pp

=== Overlap Tier3 vs Tier2 ===
rows in Tier2-high store-months: 0
rows flagged by Tier3 @0.02: 61670
rows caught by BOTH: 0 (0.0% of Tier3 drops)
Tier3-only